# Model Terbaik Anti-Overfitting Prediksi Kurs USD/IDR (Title-Only DeBERTa + FinBERT, 45 Fitur)

Notebook ini merupakan **pipeline utuh (end-to-end) versi anti-overfitting** dengan **embedding judul berbasis DeBERTa-v3-base saja** dan **sentimen berbasis FinBERT saja** (45 fitur Stage 2):

### Perbaikan Anti-Overfitting Utama (mengatasi gap Train 62.3% vs Val 48.7%):
1. **Reduksi Kapasitas Drastis (`RegularizedSeqLSTM`)**: mengganti Bi-LSTM berkapasitas ~58.500 parameter (rasio 72:1 terhadap 811 sampel Train) menjadi arsitektur **Bottleneck Projection (`45 -> 16`) + Compact LSTM (`hidden_dim = 16–24`) + Residual Skip Connection** (~3.000–5.000 parameter saja).
2. **Regularisasi Input & Augmentasi Train**: penambahan **Gaussian Noise (`noise_std=0.08`)** dan **Input Feature Dropout (`0.25–0.35`)** + **Head Dropout (`0.45`)** agar model tidak menghafal noise harian.
3. **Loss, Optimizer, & Scheduler Stabil**: **`HuberLoss`** + penalti arah ringan (`dir_lambda=0.05`), **`AdamW(lr=4e-4, weight_decay=2e-2 s/d 5e-2)`**, **`CosineAnnealingLR`**, serta **`min_epochs=10, patience=15`** agar tidak berhenti prematur di epoch 1–3.
4. **Validation-Only Gap-Penalized Selection + 5-Seed Ensemble**: pemilihan konfigurasi pada Validation Set menghukum selisih `Train - Val BalAcc`, lalu prediksi dirata-ratakan dari **5 random seed** (`42, 7, 123, 2024, 99`) untuk meredam varians antar-seed.

Seluruh pipeline menerapkan **Strict Chronological Split (70% Train / 15% Val / 15% Test)** sejak awal untuk menjamin **zero data leakage**.

## 1. Setup Lingkungan & Seed Deterministik

In [ ]:
# !pip install -q bertopic  # Aktifkan jika dijalankan di environment baru (mis. Colab/Kaggle)

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["USE_TF"] = "0"

import gc
import re
import copy
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer

from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

warnings.filterwarnings("ignore")

SEED = 42
def set_seed(seed: int = SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.titlesize": 11.5,
    "axes.titleweight": "bold",
    "axes.labelsize": 9.5,
})
sns.set_style("whitegrid")

print(f"Device aktif : {DEVICE}")
print(f"PyTorch      : {torch.__version__}")

## 2. Pemuatan Dataset & Split Kronologis di Awal (Zero Leakage)

Membaca `merged_data_clean.csv`, menghitung `rate_mid`, `log_ret`, dan target arah `y_dir` (`1 = Kurs Naik`, `0 = Kurs Turun`), lalu langsung melakukan split kronologis `70% Train / 15% Val / 15% Test` sebelum ekstraksi fitur parametrik apa pun.

In [ ]:
DATA_CANDIDATES = [
    Path("data/processed/merged_data_clean.csv"),
    Path("../data/processed/merged_data_clean.csv"),
    Path("/kaggle/input/datasets/graceee12/process-dataset/processed/merged_data_clean.csv"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Tidak menemukan data/processed/merged_data_clean.csv!")

df = pd.read_csv(DATA_PATH)
date_col = "Tanggal_dt" if "Tanggal_dt" in df.columns else "tanggal"
df[date_col] = pd.to_datetime(df[date_col])
df = df.sort_values(date_col).reset_index(drop=True)

if "rate_mid" not in df.columns:
    df["rate_mid"] = (df["Kurs Jual"] + df["Kurs Beli"]) / 2.0

df["log_ret"] = np.log(df["rate_mid"] / df["rate_mid"].shift(1))
df = df.dropna(subset=["log_ret"]).reset_index(drop=True)
df["y_dir"] = (df["log_ret"] > 0).astype(int)

def clean_text_keep_case(text) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    return re.sub(r"\s+", " ", text).strip()

df["title_clean"] = df["all_title"].fillna("").apply(clean_text_keep_case)
df["norm_clean"] = df["all_normalized_text"].fillna("").apply(clean_text_keep_case)
df["has_news"] = (df["news_count"].fillna(0) > 0).astype(int)

# SPLIT KRONOLOGIS DI AWAL (70% Train / 15% Val / 15% Test)
n_total = len(df)
train_end_idx = int(n_total * 0.70)
val_end_idx = int(n_total * 0.85)

df["split"] = "test"
df.loc[:train_end_idx - 1, "split"] = "train"
df.loc[train_end_idx:val_end_idx - 1, "split"] = "val"

train_end_date = df.loc[train_end_idx - 1, date_col]
val_end_date = df.loc[val_end_idx - 1, date_col]

is_tr = (df["split"] == "train").values
is_va = (df["split"] == "val").values
is_te = (df["split"] == "test").values

y_tr_dir = df.loc[is_tr, "y_dir"].values
y_va_dir = df.loc[is_va, "y_dir"].values
y_te_dir = df.loc[is_te, "y_dir"].values

print(f"Data path: {DATA_PATH}")
print(f"Dataset  : {n_total:,} hari perdagangan valid")
for sp, m in [("Train", is_tr), ("Val", is_va), ("Test", is_te)]:
    sub = df.loc[m]
    print(
        f"  {sp:5s}: {len(sub):4d} hari | {sub[date_col].min().date()} s/d {sub[date_col].max().date()} "
        f"| Naik={sub['y_dir'].mean()*100:.1f}% | Ada Berita={sub['has_news'].mean()*100:.1f}%"
    )

## 3. Embedding Judul Only (Frozen Layer): DeBERTa-v3-base + PCA

- Ekstraksi embedding judul berita (`title_clean`, `max_length=48`) hanya menggunakan **`microsoft/deberta-v3-base`** dengan bobot pre-trained **frozen**.
- `StandardScaler` + `PCA(n_components=8)` di-fit **hanya pada Train set (`is_tr`)** lalu ditransformasikan ke Val dan Test.

In [ ]:
def extract_daily_title_embeddings(model_name: str, unique_titles: list[str], title_lists, batch_size: int = 64, max_len: int = 48):
    print(f"Memuat frozen model untuk embedding judul: {model_name} ...")
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModel.from_pretrained(model_name).to(DEVICE)
    mdl.eval()
    for p in mdl.parameters():
        p.requires_grad = False

    title_emb_map = {}
    with torch.no_grad():
        for i in range(0, len(unique_titles), batch_size):
            batch_txt = unique_titles[i : i + batch_size]
            enc = tok(batch_txt, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(DEVICE)
            out = mdl(**enc).last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1).float()
            emb = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
            emb_np = emb.cpu().numpy()
            for t_str, vec in zip(batch_txt, emb_np):
                title_emb_map[t_str] = vec

    hidden_dim = next(iter(title_emb_map.values())).shape[0]
    daily_emb = np.zeros((len(title_lists), hidden_dim), dtype=np.float32)
    for row_idx, t_list in enumerate(title_lists):
        if t_list:
            daily_emb[row_idx] = np.mean([title_emb_map[t] for t in t_list], axis=0)

    del mdl, tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return daily_emb


title_lists = [[p.strip() for p in s.split(" | ") if p.strip()] for s in df["title_clean"]]
unique_titles = sorted({t for sub in title_lists for t in sub})
print(f"Total judul unik: {len(unique_titles):,} | Hari dengan berita: {int(df['has_news'].sum())}/{len(df)}")

DEBERTA_MODEL_NAME = "microsoft/deberta-v3-base"
daily_deberta_title = extract_daily_title_embeddings(DEBERTA_MODEL_NAME, unique_titles, title_lists, batch_size=64, max_len=48)
print(f"DeBERTa Title daily matrix: {daily_deberta_title.shape}")

N_PCA_DEB = 8
sc_deb = StandardScaler().fit(daily_deberta_title[is_tr])
pca_deb = PCA(n_components=N_PCA_DEB, random_state=SEED).fit(sc_deb.transform(daily_deberta_title[is_tr]))
deb_pca_feats = pca_deb.transform(sc_deb.transform(daily_deberta_title))
deberta_pca_cols = [f"deberta_pca_{i}" for i in range(N_PCA_DEB)]
for i, c in enumerate(deberta_pca_cols):
    df[c] = deb_pca_feats[:, i].astype(np.float32)

print(f"PCA Var Explained -> DeBERTa-Title (8): {pca_deb.explained_variance_ratio_.sum():.3f}")

## 4. Analisis Sentimen Finansial (Frozen FinBERT)

Logika kurs **USD/IDR**:
- Berita negatif $\rightarrow$ `kurs_pressure = P(neg) - P(pos) > 0` (USD/IDR naik / Rupiah melemah).
- Berita positif $\rightarrow$ `kurs_pressure < 0` (USD/IDR turun / Rupiah menguat).

Sentimen diekstrak dari **`ProsusAI/finbert`** (frozen) untuk **Stage 1 (`LogReg`)**, fitur interaksi `Sent×Arah`, serta `GRP_FINBERT_SENT` di Stage 2.

In [ ]:
def extract_daily_sentiment(model_name: str, unique_titles: list[str], title_lists, prefix: str, batch_size: int = 64, max_len: int = 64):
    print(f"Memuat frozen sentiment model: {model_name} ...")
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_name).to(DEVICE)
    mdl.eval()
    for p in mdl.parameters():
        p.requires_grad = False

    id2lbl = {int(k): v.lower() for k, v in mdl.config.id2label.items()}
    print(f"  -> id2label ({prefix}): {id2lbl}")
    pos_idx = next(k for k, v in id2lbl.items() if ("pos" in v) or ("bull" in v))
    neg_idx = next(k for k, v in id2lbl.items() if ("neg" in v) or ("bear" in v))
    neu_idx = next(k for k, v in id2lbl.items() if "neu" in v)

    sent_map = {}
    with torch.no_grad():
        for i in range(0, len(unique_titles), batch_size):
            batch_txt = unique_titles[i : i + batch_size]
            enc = tok(batch_txt, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(DEVICE)
            logits = mdl(**enc).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            for t_str, pr in zip(batch_txt, probs):
                sent_map[t_str] = (float(pr[pos_idx]), float(pr[neg_idx]), float(pr[neu_idx]))

    del mdl, tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    daily_pos = np.zeros(len(title_lists), dtype=np.float32)
    daily_neg = np.zeros(len(title_lists), dtype=np.float32)
    daily_neu = np.zeros(len(title_lists), dtype=np.float32)
    for row_idx, t_list in enumerate(title_lists):
        if t_list:
            arr = np.array([sent_map[t] for t in t_list], dtype=np.float32)
            daily_pos[row_idx], daily_neg[row_idx], daily_neu[row_idx] = arr.mean(axis=0)
        else:
            daily_neu[row_idx] = 1.0

    kp = (daily_neg - daily_pos).astype(np.float32)
    sp = (daily_pos - daily_neg).astype(np.float32)
    smag = np.abs(kp).astype(np.float32)
    srat = (daily_neg / (daily_pos + daily_neg + 1e-6)).astype(np.float32)

    return {
        f"{prefix}_pos": daily_pos,
        f"{prefix}_neg": daily_neg,
        f"{prefix}_neu": daily_neu,
        f"{prefix}_kurs_pressure": kp,
        f"{prefix}_sent_polarity": sp,
        f"{prefix}_sent_magnitude": smag,
        f"{prefix}_sent_ratio": srat,
    }


fin_dict = extract_daily_sentiment("ProsusAI/finbert", unique_titles, title_lists, prefix="finbert", batch_size=64, max_len=64)
for k, v in fin_dict.items():
    df[k] = v

fin_cont_cols = [
    "finbert_kurs_pressure", "finbert_sent_polarity", "finbert_sent_magnitude",
    "finbert_sent_ratio", "finbert_pos", "finbert_neg", "finbert_neu", "has_news",
]

PRIMARY_SENT_SRC = "FinBERT"
df["kurs_pressure"]  = df["finbert_kurs_pressure"]
df["sent_polarity"]  = df["finbert_sent_polarity"]
df["sent_magnitude"] = df["finbert_sent_magnitude"]
df["sent_ratio"]     = df["finbert_sent_ratio"]
print(f"Sumber sentimen: {PRIMARY_SENT_SRC}")

## 5. Topic Modelling BERTopic + Reduksi Dimensi (PCA-5 & UMAP-2D, Fit di Train Saja)

In [ ]:
try:
    st_model = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE, local_files_only=True)
except Exception:
    st_model = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)
for p in st_model.parameters():
    p.requires_grad = False
st_model.eval()

daily_topic_docs = [
    f"{t}. {b[:500]}".strip() if hn == 1 else ""
    for t, b, hn in zip(df["title_clean"], df["norm_clean"], df["has_news"])
]
news_mask = df["has_news"].values == 1
tr_news_mask = is_tr & news_mask
news_docs_all = [d for d, m in zip(daily_topic_docs, news_mask) if m]

with torch.no_grad():
    news_embs_all = st_model.encode(news_docs_all, batch_size=64, show_progress_bar=False, normalize_embeddings=True)

emb_minilm_full = np.zeros((len(df), news_embs_all.shape[1]), dtype=np.float32)
emb_minilm_full[news_mask] = news_embs_all

tr_docs = [daily_topic_docs[i] for i in np.where(tr_news_mask)[0]]
tr_embs = emb_minilm_full[tr_news_mask]

umap_bt = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=SEED)
hdbscan_bt = HDBSCAN(min_cluster_size=15, min_samples=5, metric="euclidean", cluster_selection_method="eom", prediction_data=True)
vec_bt = CountVectorizer(stop_words="english", min_df=3, ngram_range=(1, 2))

topic_model = BERTopic(
    embedding_model=st_model,
    umap_model=umap_bt,
    hdbscan_model=hdbscan_bt,
    vectorizer_model=vec_bt,
    calculate_probabilities=True,
    verbose=False,
)
tr_topics, _ = topic_model.fit_transform(tr_docs, tr_embs)
bt_info = topic_model.get_topic_info()
print(f"BERTopic di-fit pada {len(tr_docs)} hari Train berberita -> {int((bt_info['Topic'] != -1).sum())} topik.")
display(bt_info.head(10))

all_news_topics, all_news_probs = topic_model.transform(news_docs_all, emb_minilm_full[news_mask])
all_news_probs = np.asarray(all_news_probs)
if all_news_probs.ndim == 1:
    uniq_t = sorted([t for t in set(tr_topics) if t != -1])
    oh = np.zeros((len(all_news_topics), max(len(uniq_t), 1)), dtype=np.float32)
    tmap = {t: j for j, t in enumerate(uniq_t)}
    for r_i, tv in enumerate(all_news_topics):
        if tv in tmap:
            oh[r_i, tmap[tv]] = 1.0
    all_news_probs = oh

topic_prob_full = np.zeros((len(df), all_news_probs.shape[1]), dtype=np.float32)
topic_prob_full[news_mask] = all_news_probs
df["bt_topic"] = -2
df.loc[news_mask, "bt_topic"] = all_news_topics

TOPIC_PCA_DIM = min(5, topic_prob_full.shape[1])
sc_top = StandardScaler().fit(topic_prob_full[is_tr])
pca_top = PCA(n_components=TOPIC_PCA_DIM, random_state=SEED).fit(sc_top.transform(topic_prob_full[is_tr]))
top_pca = pca_top.transform(sc_top.transform(topic_prob_full))
topic_pca_cols = [f"topic_pca_{i}" for i in range(TOPIC_PCA_DIM)]
for i, c in enumerate(topic_pca_cols):
    df[c] = top_pca[:, i]

umap_2d = UMAP(n_neighbors=15, n_components=2, min_dist=0.1, metric="cosine", random_state=SEED).fit(tr_embs)
umap_2d_news = umap_2d.transform(emb_minilm_full[news_mask])
df["topic_umap_0"], df["topic_umap_1"] = 0.0, 0.0
df.loc[news_mask, "topic_umap_0"] = umap_2d_news[:, 0]
df.loc[news_mask, "topic_umap_1"] = umap_2d_news[:, 1]
topic_umap_cols = ["topic_umap_0", "topic_umap_1"]

del st_model
gc.collect()

## 6. Stage 1 Classifier Arah Return (`LogReg(C=0.1, d=8)`) + 5-Fold OOF

Menggunakan embedding judul **DeBERTa-v3-base saja (`deberta_pca_0..7`)**, fitur sentimen FinBERT, dan fitur topik BERTopic (total 23 fitur) dengan **5-Fold Out-Of-Fold (OOF)** di Train Set.

In [ ]:
stage1_cols = deberta_pca_cols[:8] + fin_cont_cols + topic_pca_cols + topic_umap_cols  # 23 fitur
def make_best_s1():
    return LogisticRegression(C=0.1, class_weight="balanced", max_iter=1000, random_state=SEED)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_tr_prob = np.zeros(len(y_tr_dir), dtype=np.float32)
for tr_f, ho_f in skf.split(df.loc[is_tr, stage1_cols], y_tr_dir):
    sc_f = StandardScaler().fit(df.loc[is_tr, stage1_cols].iloc[tr_f])
    clf_f = make_best_s1().fit(sc_f.transform(df.loc[is_tr, stage1_cols].iloc[tr_f]), y_tr_dir[tr_f])
    oof_tr_prob[ho_f] = clf_f.predict_proba(sc_f.transform(df.loc[is_tr, stage1_cols].iloc[ho_f]))[:, 1]

sc_s1 = StandardScaler().fit(df.loc[is_tr, stage1_cols])
final_s1 = make_best_s1().fit(sc_s1.transform(df.loc[is_tr, stage1_cols]), y_tr_dir)
p_s1_va = final_s1.predict_proba(sc_s1.transform(df.loc[is_va, stage1_cols]))[:, 1]
p_s1_te = final_s1.predict_proba(sc_s1.transform(df.loc[is_te, stage1_cols]))[:, 1]

df.loc[is_tr, "pred_dir_prob"] = oof_tr_prob
df.loc[is_va, "pred_dir_prob"] = p_s1_va
df.loc[is_te, "pred_dir_prob"] = p_s1_te
df["pred_dir_class"] = (df["pred_dir_prob"] >= 0.5).astype(int)
df["pred_dir_signed"] = 2.0 * df["pred_dir_prob"] - 1.0

s1_eval_df = pd.DataFrame([
    {"Split": "Train (5-Fold OOF)", "Acc (%)": round(accuracy_score(y_tr_dir, (oof_tr_prob >= 0.5).astype(int)) * 100, 2),
     "BalAcc (%)": round(balanced_accuracy_score(y_tr_dir, (oof_tr_prob >= 0.5).astype(int)) * 100, 2),
     "ROC-AUC": round(roc_auc_score(y_tr_dir, oof_tr_prob), 4)},
    {"Split": "Validation", "Acc (%)": round(accuracy_score(y_va_dir, (p_s1_va >= 0.5).astype(int)) * 100, 2),
     "BalAcc (%)": round(balanced_accuracy_score(y_va_dir, (p_s1_va >= 0.5).astype(int)) * 100, 2),
     "ROC-AUC": round(roc_auc_score(y_va_dir, p_s1_va), 4)},
    {"Split": "Test", "Acc (%)": round(accuracy_score(y_te_dir, (p_s1_te >= 0.5).astype(int)) * 100, 2),
     "BalAcc (%)": round(balanced_accuracy_score(y_te_dir, (p_s1_te >= 0.5).astype(int)) * 100, 2),
     "ROC-AUC": round(roc_auc_score(y_te_dir, p_s1_te), 4)},
])
print(f"Stage 1 Model: LogReg(C=0.1, d=8) ({len(stage1_cols)} fitur)")
display(s1_eval_df)

## 7. Fitur Interaksi `Sentimen × Arah` (Q70 & Q90 Train)

In [ ]:
tr_mags = df.loc[is_tr & (df["has_news"] == 1), "sent_magnitude"].values
HIGH_SENT_THRES = float(np.quantile(tr_mags, 0.70))
EXTREME_SENT_THRES = float(np.quantile(tr_mags, 0.90))
print(f"Ambang Sentimen Tinggi  (Q70 Train): {HIGH_SENT_THRES:.4f}")
print(f"Ambang Sentimen Ekstrem (Q90 Train): {EXTREME_SENT_THRES:.4f}")

df["sent_x_dir"] = df["kurs_pressure"] * df["pred_dir_signed"]
df["sent_mag_x_dir"] = df["sent_magnitude"] * df["pred_dir_signed"]
df["extreme_amp_x_dir"] = np.sign(df["sent_x_dir"]) * (np.abs(df["sent_x_dir"]) ** 2)

is_high = df["sent_magnitude"] >= HIGH_SENT_THRES
is_ext = df["sent_magnitude"] >= EXTREME_SENT_THRES

df["flag_naik_sent_tinggi"] = ((df["pred_dir_class"] == 1) & is_high).astype(int)
df["flag_turun_sent_tinggi"] = ((df["pred_dir_class"] == 0) & is_high).astype(int)
df["flag_naik_ekstrem"] = ((df["pred_dir_class"] == 1) & is_ext).astype(int)
df["flag_turun_ekstrem"] = ((df["pred_dir_class"] == 0) & is_ext).astype(int)

df["extreme_turun_pressure"] = df["flag_turun_ekstrem"] * (-df["sent_magnitude"])
df["extreme_naik_pressure"] = df["flag_naik_ekstrem"] * (df["sent_magnitude"])

df["sent_dir_5tier"] = (
    df["flag_naik_sent_tinggi"] + df["flag_naik_ekstrem"]
    - df["flag_turun_sent_tinggi"] - df["flag_turun_ekstrem"]
)

## 8. Fitur Time Series Kausal ($\le t-1$) & Seleksi 45 Fitur (DeBERTa Title Only)

Menyusun 14 fitur *time series* kausal dan menggabungkannya dengan **embedding judul DeBERTa** + **FinBERT** + **`Sent×Arah`** + **BERTopic** (total **45 fitur**) untuk input sekuensial `RegularizedSeqLSTM` (`seq_len = 10`).

In [ ]:
df["ret_lag1"] = df["log_ret"].shift(1)
df["ret_lag2"] = df["log_ret"].shift(2)
df["ret_lag3"] = df["log_ret"].shift(3)
df["ret_lag5"] = df["log_ret"].shift(5)
df["vol_5d"] = df["log_ret"].shift(1).rolling(5).std()
df["vol_10d"] = df["log_ret"].shift(1).rolling(10).std()

for w in [5, 10, 20]:
    ma_w = df["rate_mid"].shift(1).rolling(w).mean()
    df[f"dist_ma{w}"] = df["rate_mid"].shift(1) / ma_w - 1.0

delta = df["rate_mid"].diff().shift(1)
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
df["rsi_14"] = (100.0 - (100.0 / (1.0 + gain / (loss + 1e-9)))) / 100.0

consecs, cur = [], 0
for r in df["log_ret"]:
    cur = (cur + 1 if cur > 0 else 1) if r > 0 else ((cur - 1 if cur < 0 else -1) if r < 0 else 0)
    consecs.append(cur)
df["consec_up_days"] = pd.Series(consecs, index=df.index).shift(1)
df["spread_lag1"] = ((df["Kurs Jual"] - df["Kurs Beli"]) / df["rate_mid"]).shift(1)
dow = df[date_col].dt.dayofweek
df["dow_sin"] = np.sin(2 * np.pi * dow / 5.0)
df["dow_cos"] = np.cos(2 * np.pi * dow / 5.0)

GRP_TS = [
    "ret_lag1", "ret_lag2", "ret_lag3", "ret_lag5",
    "vol_5d", "vol_10d", "dist_ma5", "dist_ma10", "dist_ma20",
    "rsi_14", "consec_up_days", "spread_lag1", "dow_sin", "dow_cos",
]
GRP_DEBERTA_TITLE = deberta_pca_cols[:4]
GRP_FINBERT_SENT = [
    "finbert_kurs_pressure", "finbert_sent_magnitude", "finbert_pos", "finbert_neg", "finbert_neu",
]
GRP_SENT_INTER = [
    "kurs_pressure", "sent_magnitude", "has_news",
    "pred_dir_prob", "pred_dir_signed",
    "sent_x_dir", "sent_mag_x_dir", "extreme_amp_x_dir",
    "flag_naik_sent_tinggi", "flag_turun_sent_tinggi",
    "flag_naik_ekstrem", "flag_turun_ekstrem",
    "extreme_turun_pressure", "extreme_naik_pressure", "sent_dir_5tier",
]
GRP_BERTOPIC = topic_pca_cols + topic_umap_cols

BEST_COLS = GRP_TS + GRP_DEBERTA_TITLE + GRP_FINBERT_SENT + GRP_SENT_INTER + GRP_BERTOPIC
assert len(BEST_COLS) == 45, f"Expected 45 fitur, got {len(BEST_COLS)}"

df_model = df.dropna(subset=BEST_COLS + ["log_ret"]).reset_index(drop=True)
m_tr = (df_model[date_col] <= train_end_date).values
m_va = ((df_model[date_col] > train_end_date) & (df_model[date_col] <= val_end_date)).values
m_te = (df_model[date_col] > val_end_date).values

print(f"Baris valid Stage 2: {len(df_model):,} | Train={m_tr.sum()}, Val={m_va.sum()}, Test={m_te.sum()}")
print(
    f"Total Fitur Terpilih: {len(BEST_COLS)} "
    f"(TS={len(GRP_TS)}, DeBERTa-Title={len(GRP_DEBERTA_TITLE)}, "
    f"FinBERT={len(GRP_FINBERT_SENT)}, Sent×Arah={len(GRP_SENT_INTER)}, BERTopic={len(GRP_BERTOPIC)})"
)

def build_sequences(frame: pd.DataFrame, feature_cols: list[str], seq_len: int):
    sc_x = StandardScaler().fit(frame.loc[m_tr, feature_cols])
    X_scaled = sc_x.transform(frame[feature_cols]).astype(np.float32)
    y_raw = frame["log_ret"].values.astype(np.float32)
    mu_y = float(y_raw[m_tr].mean())
    std_y = float(y_raw[m_tr].std() + 1e-9)
    y_scaled = ((y_raw - mu_y) / std_y).astype(np.float32)

    X_seqs, y_seqs_s, y_seqs_raw, seq_splits, seq_dates, prev_rates, curr_rates = [], [], [], [], [], [], []
    split_arr = np.where(m_tr, "train", np.where(m_va, "val", "test"))
    dates_arr = frame[date_col].values
    rates_arr = frame["rate_mid"].values

    for t in range(seq_len - 1, len(frame)):
        X_seqs.append(X_scaled[t - seq_len + 1 : t + 1])
        y_seqs_s.append(y_scaled[t])
        y_seqs_raw.append(y_raw[t])
        seq_splits.append(split_arr[t])
        seq_dates.append(dates_arr[t])
        prev_rates.append(rates_arr[t - 1])
        curr_rates.append(rates_arr[t])

    return {
        "X": np.array(X_seqs, dtype=np.float32),
        "y_s": np.array(y_seqs_s, dtype=np.float32),
        "y_raw": np.array(y_seqs_raw, dtype=np.float32),
        "split": np.array(seq_splits),
        "dates": pd.to_datetime(seq_dates),
        "prev_rate": np.array(prev_rates, dtype=np.float64),
        "curr_rate": np.array(curr_rates, dtype=np.float64),
        "mu_y": mu_y,
        "std_y": std_y,
        "scaler_x": sc_x,
    }

BEST_SEQ_LEN = 10
seq_data = build_sequences(df_model, BEST_COLS, seq_len=BEST_SEQ_LEN)
sp = seq_data["split"]
print(f"Shape sequence X: {seq_data['X'].shape} | Train={(sp=='train').sum()}, Val={(sp=='val').sum()}, Test={(sp=='test').sum()}")

## 9. Arsitektur Anti-Overfitting (`RegularizedSeqLSTM`), Seleksi di Validation, & 5-Seed Ensemble

Perubahan kunci untuk mencegah *overfitting*:
- **Kapasitas Ringkas (~3.000–5.000 parameter, turun >90% dari 58.500):** `LayerNorm -> Gaussian Noise -> Input Dropout -> Bottleneck Linear(45 -> 16) -> LSTM(hidden_dim=16/24) + Residual Skip`.
- **Training Stabil:** `HuberLoss(delta=1.0)` + `dir_lambda=0.05`, `AdamW(lr=4e-4, weight_decay=2e-2..5e-2)`, `CosineAnnealingLR`, dan `min_epochs=10, patience=15`.
- **Anti-Overfit Val Tuning + 5-Seed Ensemble:** memilih konfigurasi di Validation Set dengan penalti gap `max(0, Train BalAcc - Val BalAcc)`, lalu menggabungkan prediksi **5 seed (`42, 7, 123, 2024, 99`)** untuk menstabilkan Test Set.

In [ ]:
class RegularizedSeqLSTM(nn.Module):
    def __init__(self, input_dim: int, proj_dim: int = 16, hidden_dim: int = 16,
                 input_dropout: float = 0.30, dropout: float = 0.45,
                 noise_std: float = 0.08, bidirectional: bool = False):
        super().__init__()
        self.noise_std = noise_std
        self.in_norm = nn.LayerNorm(input_dim)
        self.in_drop = nn.Dropout(input_dropout)
        self.proj = nn.Sequential(
            nn.Linear(input_dim, proj_dim),
            nn.Tanh(),
            nn.Dropout(dropout * 0.5),
        )
        self.lstm = nn.LSTM(
            input_size=proj_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=bidirectional,
        )
        out_dim = hidden_dim * (2 if bidirectional else 1)
        self.head = nn.Sequential(
            nn.LayerNorm(out_dim),
            nn.Dropout(dropout),
            nn.Linear(out_dim, 1),
        )
        self.skip = nn.Linear(proj_dim, 1)

    def forward(self, x):
        x = self.in_norm(x)
        if self.training and self.noise_std > 0:
            x = x + torch.randn_like(x) * self.noise_std
        x = self.in_drop(x)
        p = self.proj(x)
        out, _ = self.lstm(p)
        return (self.head(out[:, -1, :]) + 0.3 * self.skip(p[:, -1, :])).squeeze(-1)


def train_pytorch_regressor(model: nn.Module, X_tr_np, y_tr_s_np, X_va_np, y_va_s_np, X_te_np,
                            mu_y: float, std_y: float, lr: float = 4e-4, weight_decay: float = 3e-2,
                            dir_lambda: float = 0.05, batch_size: int = 32, epochs: int = 60,
                            patience: int = 15, min_epochs: int = 10, seed: int = SEED):
    set_seed(seed)
    X_tr = torch.tensor(X_tr_np, dtype=torch.float32)
    y_tr = torch.tensor(y_tr_s_np, dtype=torch.float32)
    X_tr_dev = X_tr.to(DEVICE)
    X_va = torch.tensor(X_va_np, dtype=torch.float32).to(DEVICE)
    y_va = torch.tensor(y_va_s_np, dtype=torch.float32).to(DEVICE)
    X_te = torch.tensor(X_te_np, dtype=torch.float32).to(DEVICE)

    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    huber = nn.HuberLoss(delta=1.0)

    def composite_loss(pred_s, target_s):
        reg_loss = huber(pred_s, target_s)
        zero_s = -mu_y / std_y
        sign_hinge = torch.relu(-(pred_s - zero_s) * torch.sign(target_s - zero_s)).mean()
        return reg_loss + dir_lambda * sign_hinge

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    wait = 0
    best_ep = 1
    tr_hist, va_hist = [], []

    for ep in range(1, epochs + 1):
        model.train()
        ep_losses = []
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = composite_loss(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_losses.append(float(loss.item()))
        sched.step()

        model.eval()
        with torch.no_grad():
            va_loss = float(composite_loss(model(X_va), y_va).item())

        tr_hist.append(float(np.mean(ep_losses)))
        va_hist.append(va_loss)

        if va_loss < best_val_loss - 1e-6:
            best_val_loss = va_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
            best_ep = ep
        else:
            wait += 1
            if ep >= min_epochs and wait >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred_tr_raw = model(X_tr_dev).cpu().numpy() * std_y + mu_y
        pred_va_raw = model(X_va).cpu().numpy() * std_y + mu_y
        pred_te_raw = model(X_te).cpu().numpy() * std_y + mu_y

    return {
        "model": model,
        "pred_tr": pred_tr_raw,
        "pred_va": pred_va_raw,
        "pred_te": pred_te_raw,
        "tr_hist": tr_hist,
        "va_hist": va_hist,
        "best_epoch": best_ep,
    }


def eval_reg(name: str, y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    dt, dp = (y_true > 0).astype(int), (y_pred > 0).astype(int)
    return {
        "Model / Split": name,
        "RMSE": round(rmse, 6),
        "MAE": round(mae, 6),
        "R2": round(r2, 4),
        "DirAcc (%)": round(float(accuracy_score(dt, dp) * 100), 2),
        "BalAcc (%)": round(float(balanced_accuracy_score(dt, dp) * 100), 2),
        "F1-Macro (%)": round(float(f1_score(dt, dp, average="macro") * 100), 2),
    }


tr_m, va_m, te_m = sp == "train", sp == "val", sp == "test"
y_tr_true = seq_data["y_raw"][tr_m]
y_va_true = seq_data["y_raw"][va_m]
y_te_true = seq_data["y_raw"][te_m]
mu_train_raw = float(y_tr_true.mean())

# 1. Tuning Konfigurasi Ter-regularisasi pada Validation Set (Menghukum Gap Train-Val!)
reg_grid = [
    {"proj_dim": 16, "hidden_dim": 16, "bidirectional": False, "input_dropout": 0.25, "dropout": 0.45, "noise_std": 0.08, "weight_decay": 2e-2, "lr": 4e-4, "dir_lambda": 0.05},
    {"proj_dim": 16, "hidden_dim": 16, "bidirectional": False, "input_dropout": 0.35, "dropout": 0.50, "noise_std": 0.10, "weight_decay": 4e-2, "lr": 4e-4, "dir_lambda": 0.05},
    {"proj_dim": 16, "hidden_dim": 24, "bidirectional": False, "input_dropout": 0.30, "dropout": 0.50, "noise_std": 0.08, "weight_decay": 3e-2, "lr": 3e-4, "dir_lambda": 0.05},
    {"proj_dim": 12, "hidden_dim": 12, "bidirectional": True,  "input_dropout": 0.30, "dropout": 0.45, "noise_std": 0.08, "weight_decay": 3e-2, "lr": 4e-4, "dir_lambda": 0.05},
    {"proj_dim": 16, "hidden_dim": 16, "bidirectional": True,  "input_dropout": 0.35, "dropout": 0.50, "noise_std": 0.10, "weight_decay": 5e-2, "lr": 3e-4, "dir_lambda": 0.05},
    {"proj_dim": 16, "hidden_dim": 16, "bidirectional": False, "input_dropout": 0.30, "dropout": 0.45, "noise_std": 0.08, "weight_decay": 3e-2, "lr": 4e-4, "dir_lambda": 0.00},
]

tune_rows = []
for cid, cfg in enumerate(reg_grid):
    mdl = RegularizedSeqLSTM(
        input_dim=len(BEST_COLS),
        proj_dim=cfg["proj_dim"],
        hidden_dim=cfg["hidden_dim"],
        input_dropout=cfg["input_dropout"],
        dropout=cfg["dropout"],
        noise_std=cfg["noise_std"],
        bidirectional=cfg["bidirectional"],
    )
    n_params = sum(p.numel() for p in mdl.parameters())
    r = train_pytorch_regressor(
        mdl, seq_data["X"][tr_m], seq_data["y_s"][tr_m],
        seq_data["X"][va_m], seq_data["y_s"][va_m],
        seq_data["X"][te_m], seq_data["mu_y"], seq_data["std_y"],
        lr=cfg["lr"], weight_decay=cfg["weight_decay"], dir_lambda=cfg["dir_lambda"],
        batch_size=32, epochs=60, patience=15, min_epochs=10, seed=SEED,
    )
    m_tr_e = eval_reg("tr", y_tr_true, r["pred_tr"])
    m_va_e = eval_reg("va", y_va_true, r["pred_va"])
    gap_bal = max(0.0, m_tr_e["BalAcc (%)"] - m_va_e["BalAcc (%)"])
    tune_rows.append({
        "cfg_id": cid, **cfg, "n_params": n_params, "best_ep": r["best_epoch"],
        "Train Bal (%)": m_tr_e["BalAcc (%)"], "Val Bal (%)": m_va_e["BalAcc (%)"],
        "Gap (Tr-Val)": round(m_tr_e["BalAcc (%)"] - m_va_e["BalAcc (%)"], 2),
        "Val RMSE": m_va_e["RMSE"], "Val R2": m_va_e["R2"],
        "_gap_pen": gap_bal,
    })

tune_df = pd.DataFrame(tune_rows)
tune_df["Anti-Overfit Val Score"] = (
    tune_df["Val Bal (%)"]
    - 0.35 * tune_df["_gap_pen"]
    - 2000.0 * (tune_df["Val RMSE"] - tune_df["Val RMSE"].min())
).round(3)
tune_df = tune_df.sort_values("Anti-Overfit Val Score", ascending=False).reset_index(drop=True)
print("=== HASIL TUNING ANTI-OVERFITTING DI VALIDATION SET ===")
display(tune_df.drop(columns=["_gap_pen"]))

best_cid = int(tune_df.iloc[0]["cfg_id"])
BEST_CFG = {
    "seq_len": BEST_SEQ_LEN,
    **reg_grid[best_cid],
    "batch_size": 32,
    "epochs": 60,
    "patience": 15,
    "min_epochs": 10,
}
print(f"\nKonfigurasi Terpilih (cfg_id={best_cid}, params={int(tune_df.iloc[0]['n_params'])}): {BEST_CFG}")

# 2. Latih 5-Seed Ensemble dari Konfigurasi Terbaik
ENSEMBLE_SEEDS = (42, 7, 123, 2024, 99)
seed_runs = []
seed_summary_rows = []
for s in ENSEMBLE_SEEDS:
    mdl_s = RegularizedSeqLSTM(
        input_dim=len(BEST_COLS),
        proj_dim=BEST_CFG["proj_dim"],
        hidden_dim=BEST_CFG["hidden_dim"],
        input_dropout=BEST_CFG["input_dropout"],
        dropout=BEST_CFG["dropout"],
        noise_std=BEST_CFG["noise_std"],
        bidirectional=BEST_CFG["bidirectional"],
    )
    r_s = train_pytorch_regressor(
        mdl_s, seq_data["X"][tr_m], seq_data["y_s"][tr_m],
        seq_data["X"][va_m], seq_data["y_s"][va_m],
        seq_data["X"][te_m], seq_data["mu_y"], seq_data["std_y"],
        lr=BEST_CFG["lr"], weight_decay=BEST_CFG["weight_decay"], dir_lambda=BEST_CFG["dir_lambda"],
        batch_size=BEST_CFG["batch_size"], epochs=BEST_CFG["epochs"],
        patience=BEST_CFG["patience"], min_epochs=BEST_CFG["min_epochs"], seed=s,
    )
    seed_runs.append(r_s)
    mt = eval_reg("tr", y_tr_true, r_s["pred_tr"])
    mv = eval_reg("va", y_va_true, r_s["pred_va"])
    mte = eval_reg("te", y_te_true, r_s["pred_te"])
    seed_summary_rows.append({
        "Seed": s, "Best Epoch": r_s["best_epoch"],
        "Train Bal (%)": mt["BalAcc (%)"], "Val Bal (%)": mv["BalAcc (%)"],
        "Gap (Tr-Val)": round(mt["BalAcc (%)"] - mv["BalAcc (%)"], 2),
        "Test Bal (%)": mte["BalAcc (%)"], "Test Dir (%)": mte["DirAcc (%)"],
        "Test RMSE": mte["RMSE"], "Test R2": mte["R2"],
    })

print("\n=== STABILITAS PER SEED (ANTI-OVERFITTING CHECK) ===")
display(pd.DataFrame(seed_summary_rows).set_index("Seed"))

res_best = {
    "pred_tr": np.mean([r["pred_tr"] for r in seed_runs], axis=0),
    "pred_va": np.mean([r["pred_va"] for r in seed_runs], axis=0),
    "pred_te": np.mean([r["pred_te"] for r in seed_runs], axis=0),
    "tr_hist": seed_runs[0]["tr_hist"],
    "va_hist": seed_runs[0]["va_hist"],
    "best_epoch": seed_runs[0]["best_epoch"],
}

TEST_ROW_LABEL = "Regularized LSTM 5-Seed Ensemble (Test Set — 45 Fitur)"
eval_summary_df = pd.DataFrame([
    eval_reg("Baseline Test: Tebak Return 0", y_te_true, np.zeros_like(y_te_true)),
    eval_reg("Baseline Test: Mean Train", y_te_true, np.full_like(y_te_true, mu_train_raw)),
    eval_reg("Regularized LSTM Ensemble (Train Set)", y_tr_true, res_best["pred_tr"]),
    eval_reg("Regularized LSTM Ensemble (Validation Set)", y_va_true, res_best["pred_va"]),
    eval_reg(TEST_ROW_LABEL, y_te_true, res_best["pred_te"]),
]).set_index("Model / Split")

print("\n=== EVALUASI AKHIR MODEL ANTI-OVERFITTING (5-SEED ENSEMBLE, 45 FITUR) ===")
display(eval_summary_df)

y_te_dir_true = (y_te_true > 0).astype(int)
y_te_dir_pred = (res_best["pred_te"] > 0).astype(int)
print("\n=== CLASSIFICATION REPORT ARAH RETURN (TEST SET) ===")
print(classification_report(y_te_dir_true, y_te_dir_pred, target_names=["Turun (0)", "Naik (1)"], digits=4))

## 10. Visualisasi Hasil Akhir & Rekonstruksi Kurs USD/IDR (Test Set)

In [ ]:
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.25)

# 1. Kurva Composite Loss Train vs Val (Seed 42)
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(res_best["tr_hist"], label="Train Loss (Seed 42)", color="#1d3557", lw=1.8)
ax1.plot(res_best["va_hist"], label="Validation Loss (Seed 42)", color="#e63946", linestyle="--", lw=1.8)
ax1.axvline(max(res_best["best_epoch"] - 1, 0), color="gray", linestyle=":", lw=1.2, label=f"Best Epoch ({res_best['best_epoch']})")
ax1.set_title("Kurva Huber + Dir Loss per Epoch (RegularizedSeqLSTM)")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Composite Loss")
ax1.legend()

# 2. Confusion Matrix Prediksi Arah (Test Set — 5-Seed Ensemble)
ax2 = fig.add_subplot(gs[0, 1])
cm = confusion_matrix(y_te_dir_true, y_te_dir_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Pred Turun", "Pred Naik"],
            yticklabels=["Aktual Turun", "Aktual Naik"], ax=ax2)
ax2.set_title(
    f"Confusion Matrix Test Set | BalAcc={eval_summary_df.loc[TEST_ROW_LABEL, 'BalAcc (%)']:.2f}% "
    f"| DirAcc={eval_summary_df.loc[TEST_ROW_LABEL, 'DirAcc (%)']:.2f}%"
)

# 3. Deret Waktu Log Return Aktual vs Prediksi (Test Set)
te_dates = seq_data["dates"][te_m]
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(te_dates, y_te_true, label="Log Return Aktual", color="#222222", alpha=0.7, lw=1.1)
ax3.plot(te_dates, res_best["pred_te"], label="Prediksi 5-Seed Ensemble (45 Fitur)", color="#e63946", linestyle="--", lw=1.5)
ax3.axhline(0, color="gray", lw=0.8)
ax3.set_title("Log Return Test Set: Aktual vs Prediksi Anti-Overfitting")
ax3.tick_params(axis="x", rotation=25)
ax3.legend()

# 4. Rekonstruksi Kurs USD/IDR + Sinyal Sentimen Ekstrem (Test Set)
ax4 = fig.add_subplot(gs[1, 1])
prev_r = seq_data["prev_rate"][te_m]
curr_r = seq_data["curr_rate"][te_m]
pred_r = prev_r * np.exp(res_best["pred_te"])
ax4.plot(te_dates, curr_r, label="Kurs Aktual (rate_mid)", color="#1d3557", lw=1.7)
ax4.plot(te_dates, pred_r, label="Prediksi 5-Seed Ensemble", color="#f4a261", linestyle="--", lw=1.5)

te_align = df_model.loc[df_model[date_col].isin(te_dates)].copy()
ex_up = te_align[te_align["flag_naik_ekstrem"] == 1]
ex_dn = te_align[te_align["flag_turun_ekstrem"] == 1]
ax4.scatter(ex_up[date_col], ex_up["rate_mid"], marker="^", s=55, color="#e63946", zorder=5, label="Ekstrem Naik")
ax4.scatter(ex_dn[date_col], ex_dn["rate_mid"], marker="v", s=55, color="#2a9d8f", zorder=5, label="Ekstrem Turun")
ax4.set_title("Rekonstruksi Kurs USD/IDR Test Set & Sinyal Ekstrem")
ax4.tick_params(axis="x", rotation=25)
ax4.legend()

plt.tight_layout()
plt.show()

pred_test_df = pd.DataFrame({
    "Tanggal": te_dates,
    "Kurs Aktual": np.round(curr_r, 2),
    "Kurs Prediksi": np.round(pred_r, 2),
    "LogRet Aktual": np.round(y_te_true, 6),
    "LogRet Prediksi": np.round(res_best["pred_te"], 6),
    "Arah Aktual": np.where(y_te_dir_true == 1, "Naik", "Turun"),
    "Arah Prediksi": np.where(y_te_dir_pred == 1, "Naik", "Turun"),
})
display(pred_test_df.tail(10))

## 11. Ablation Study per Grup Fitur (Stage 2 RegularizedSeqLSTM)

Empat skenario: **Full**, **leave-one-group-out**, **single-group-only**, dan **TS + satu grup** menggunakan konfigurasi anti-overfitting terbaik (`BEST_CFG`) dengan **3-seed ensemble** (`42, 7, 123`) agar hasil ablasi stabil.

In [ ]:
GROUPS = {
    "TS (14)": GRP_TS,
    "DeBERTa-Title (4)": GRP_DEBERTA_TITLE,
    "FinBERT-Sent (5)": GRP_FINBERT_SENT,
    "Sent×Arah (15)": GRP_SENT_INTER,
    "BERTopic (7)": GRP_BERTOPIC,
}
ABL_SEEDS = (42, 7, 123)

def eval_reg_raw(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    dt, dp = (y_true > 0).astype(int), (y_pred > 0).astype(int)
    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "DirAcc (%)": float(accuracy_score(dt, dp) * 100),
        "BalAcc (%)": float(balanced_accuracy_score(dt, dp) * 100),
        "F1-Macro (%)": float(f1_score(dt, dp, average="macro") * 100),
    }

def run_cfg(cols, seeds=ABL_SEEDS):
    sd = build_sequences(df_model, cols, BEST_SEQ_LEN)
    s_ = sd["split"]
    tr, va, te = s_ == "train", s_ == "val", s_ == "test"
    yv, yt = sd["y_raw"][va], sd["y_raw"][te]
    p_va_seeds, p_te_seeds = [], []
    va_bals, te_bals = [], []
    for s in seeds:
        m = RegularizedSeqLSTM(
            input_dim=len(cols),
            proj_dim=min(BEST_CFG["proj_dim"], max(8, len(cols))),
            hidden_dim=BEST_CFG["hidden_dim"],
            input_dropout=BEST_CFG["input_dropout"],
            dropout=BEST_CFG["dropout"],
            noise_std=BEST_CFG["noise_std"],
            bidirectional=BEST_CFG["bidirectional"],
        )
        r = train_pytorch_regressor(
            m, sd["X"][tr], sd["y_s"][tr], sd["X"][va], sd["y_s"][va], sd["X"][te],
            sd["mu_y"], sd["std_y"], lr=BEST_CFG["lr"], weight_decay=BEST_CFG["weight_decay"],
            dir_lambda=BEST_CFG["dir_lambda"], batch_size=BEST_CFG["batch_size"],
            epochs=BEST_CFG["epochs"], patience=BEST_CFG["patience"], min_epochs=BEST_CFG["min_epochs"], seed=s,
        )
        p_va_seeds.append(r["pred_va"])
        p_te_seeds.append(r["pred_te"])
        va_bals.append(eval_reg_raw(yv, r["pred_va"])["BalAcc (%)"])
        te_bals.append(eval_reg_raw(yt, r["pred_te"])["BalAcc (%)"])

    ens_va_m = eval_reg_raw(yv, np.mean(p_va_seeds, axis=0))
    ens_te_m = eval_reg_raw(yt, np.mean(p_te_seeds, axis=0))
    return ens_va_m, ens_te_m, float(np.std(te_bals))

records = []
def add(name, cols, kind):
    v, t, t_sd = run_cfg(cols)
    records.append({
        "Skenario": name, "Tipe": kind, "#Fitur": len(cols),
        "Val BalAcc": v["BalAcc (%)"], "Val RMSE": v["RMSE"],
        "Test BalAcc": t["BalAcc (%)"], "Test BalAcc±sd": t_sd,
        "Test DirAcc": t["DirAcc (%)"], "Test F1-Macro": t["F1-Macro (%)"],
        "Test RMSE": t["RMSE"], "Test R2": t["R2"],
    })

add("Full (45 fitur)", BEST_COLS, "full")
for g, cols in GROUPS.items():
    keep = [c for c in BEST_COLS if c not in cols]
    add(f"– {g}", keep, "drop-one")
for g, cols in GROUPS.items():
    add(f"Only {g}", cols, "only-one")
for g, cols in GROUPS.items():
    if g.startswith("TS"):
        continue
    add(f"TS + {g}", GRP_TS + cols, "TS+one")

abl_df = pd.DataFrame(records).set_index("Skenario")
full = abl_df.loc["Full (45 fitur)"]
abl_df["Δ Test BalAcc vs Full"] = abl_df["Test BalAcc"] - full["Test BalAcc"]
abl_df["Δ Val BalAcc vs Full"] = abl_df["Val BalAcc"] - full["Val BalAcc"]
with pd.option_context("display.precision", 8, "display.float_format", None):
    display(abl_df)

d = abl_df[abl_df["Tipe"] == "drop-one"]
fig, ax = plt.subplots(figsize=(8, 3.8))
x = np.arange(len(d)); w = 0.38
ax.bar(x - w/2, d["Δ Val BalAcc vs Full"], w, label="Val", color="#1d3557")
ax.bar(x + w/2, d["Δ Test BalAcc vs Full"], w, label="Test", color="#e63946")
ax.axhline(0, color="gray", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(d.index, rotation=20, ha="right")
ax.set_ylabel("Δ BalAcc (poin) vs Full")
ax.set_title("Ablasi Leave-One-Group-Out (Ensemble 3 Seed, Regularized)")
ax.legend(); plt.tight_layout(); plt.show()